In [1]:
import xarray as xr, numpy as np, pandas as pd, os, glob
import dask.dataframe as dd
from dask.distributed import Client

In [2]:
client = Client(n_workers=14, threads_per_worker=1, memory_limit='4GB')
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 14
Total threads: 14,Total memory: 52.15 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43009,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38237,Total threads: 1
Dashboard: /proxy/45083/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:39251,


In [3]:
ehf_fpath = '/scratch/ng72/ms5578/hw_files'
nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/raw'
write_path = '/scratch/ng72/ms5578/time_series'
netcdf_files = glob.glob(os.path.join(ehf_fpath, "*.nc")) 

In [4]:
gen_df = pd.read_csv(f"{nmap_path}/nmap.csv")
gen_df.drop(gen_df.columns[[3, 1, 4, 5]], axis=1, inplace=True)
gen_df.columns = (
    gen_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^\w_]', '', regex=True)
    .str.replace('__', '_')
)
gen_df = gen_df.rename(columns={'duid': 'DUID'})

In [5]:
template_ds = xr.open_dataset(netcdf_files[0], engine='netcdf4')
lat_grid = template_ds['lat'].values
lon_grid = template_ds['lon'].values
gen_lats = gen_df['lat'].values
gen_lons = gen_df['lon'].values
lat_indices = np.abs(lat_grid[:, None] - gen_lats).argmin(axis=0)
lon_indices = np.abs(lon_grid[:, None] - gen_lons).argmin(axis=0)

In [6]:
isel_dict = {
    'lat': xr.DataArray(lat_indices, dims='points'),
    'lon': xr.DataArray(lon_indices, dims='points'),
}

all_ddfs = []

for file in netcdf_files:
    ds = xr.open_dataset(file, engine='netcdf4', chunks='auto')
    sub_ds = ds.isel(lat=isel_dict['lat'], lon=isel_dict['lon'])
    sub_ds = sub_ds.assign_coords(DUID=('points', gen_df['DUID'].values))
    df = sub_ds[['EHF_val', 'HW_EHF_avg', 'HW_EHF_peak', 'EHF_flag','tas_3d_avg','tas_3d_peak']].to_dask_dataframe().reset_index()
    all_ddfs.append(df)


full_ddf = dd.concat(all_ddfs)

In [7]:
def clean_and_process(df):
    df['time'] = dd.to_datetime(df['time'])
    df = df.replace(1.000000e+20, np.nan)
    df = df.drop(columns=['lat', 'lon', 'height', 'crs'], errors='ignore')
    df = df.sort_values(by=['DUID', 'time'])
    return df

In [8]:
def number_heatwave_days(df):
    df = df.sort_values(by='time')
    is_hw = df['EHF_flag'] == 1
    event_id = (is_hw != is_hw.shift()).cumsum()
    df['event_group'] = np.where(is_hw, event_id, pd.NA)
    df['HW_event_day'] = df.groupby('event_group').cumcount() + 1
    df.loc[df['event_group'].isna(), 'HW_event_day'] = pd.NA
    df = df.drop(columns='event_group')
    return df

In [9]:
full_ddf = clean_and_process(full_ddf)
grouped = full_ddf.groupby('DUID').apply(number_heatwave_days, meta=full_ddf)

In [10]:
df = full_ddf.compute()
df = number_heatwave_days(df)
df.to_csv(f"{write_path}/gen_hw_status.csv", index=False)
gen_df.to_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/nmap.csv', index=False)
